# M13 — Learn from Neighbors

**Objective:** understand instance-based learning through KNN behavior and distance reasoning.

This CPU-only lab uses a checked-in synthetic dataset, fixed random seeds, no secrets, no paid API and no runtime network access.


## Learning contract

Use this loop for every experiment:

**prediction → run → observation → explanation → generalization → transfer**

Do not explain a result using accuracy alone. Name the query, its neighbor identities, their distances, their votes and the feature units that defined *near*.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 13
CLASS_ORDER = ["guided", "independent"]
CLASS_TO_INT = {label: index for index, label in enumerate(CLASS_ORDER)}
CLASS_COLORS = {"guided": "#d95f02", "independent": "#1b9e77"}
pd.set_option("display.max_columns", 20)


In [ ]:
relative_data_path = Path("datasets/M13/knn_scale_cases.csv")
search_roots = [Path.cwd(), *Path.cwd().parents]
DATA_PATH = next(
    (root / relative_data_path for root in search_roots if (root / relative_data_path).is_file()),
    None,
)
assert DATA_PATH is not None, f"Could not locate {relative_data_path} from {Path.cwd()}"

data = pd.read_csv(DATA_PATH)
informative_features = ["practice_hours", "assessment_score"]
weak_feature = "interface_event_count"
target = "learning_route"

print(f"Loaded {len(data)} cases from {DATA_PATH}")
display(data.head())


In [ ]:
summary = data[informative_features + [weak_feature]].agg(["min", "max", "mean", "std"]).round(3)
class_balance = data[target].value_counts().reindex(CLASS_ORDER)
display(summary)
display(class_balance.rename("cases"))


## System map before mechanism theory

`CSV rows → selected features → fixed train/test split → optional scaler → stored training cases → query distances → k neighbors → majority vote → prediction`

KNN learns no compact boundary parameters during `fit`; the training instances are the model state. `StandardScaler` *does* learn means and scales, and must learn them from training rows only.


## Prediction checkpoint 1 — visualize

Before running the next cell, sketch where you expect `guided` and `independent` cases. Record:

- where the classes should form local neighborhoods;
- one region you expect to be ambiguous;
- whether a straight line seems sufficient.

**Your prediction:** _write here before running_.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for label in CLASS_ORDER:
    group = data[data[target] == label]
    ax.scatter(
        group["practice_hours"],
        group["assessment_score"],
        label=label,
        color=CLASS_COLORS[label],
        alpha=0.82,
        edgecolor="white",
        linewidth=0.5,
    )
ax.set(title="Local class geometry", xlabel="Practice hours", ylabel="Assessment score")
ax.legend(title="Learning route")
ax.grid(alpha=0.18)
plt.show()


## Narrow mechanism: distance and vote

For Euclidean distance, each coordinate difference is squared, summed and square-rooted. With `k=5`, the five smallest distances determine the vote. A feature measured in numerically larger units contributes more unless the feature space is deliberately transformed.


In [ ]:
train_index, test_index = train_test_split(
    data.index,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=data[target],
)
X_train = data.loc[train_index, informative_features]
X_test = data.loc[test_index, informative_features]
y_train = data.loc[train_index, target]
y_test = data.loc[test_index, target]

assert set(X_train.index).isdisjoint(X_test.index)
print("train shape:", X_train.shape, "test shape:", X_test.shape)


In [ ]:
def make_neighbor_table(reference_features, reference_labels, distances, positions):
    # Return raw feature values beside model-space distances and labels.
    table = reference_features.iloc[positions[0]].copy()
    table.insert(0, "case_id", data.loc[table.index, "case_id"])
    table["distance"] = distances[0]
    table["neighbor_label"] = reference_labels.iloc[positions[0]].to_numpy()
    return table.reset_index(drop=True)


## Prediction checkpoint 2 — query point and nearest neighbors

The query has `practice_hours=6.0` and `assessment_score=67.0`.

Before running:

1. predict its class for raw Euclidean KNN with `k=5`;
2. mark five likely neighbors on the plot;
3. predict the vote split.

**Your prediction:** _write here before running_.


In [ ]:
query = pd.DataFrame([[6.0, 67.0]], columns=informative_features)
raw_knn = KNeighborsClassifier(n_neighbors=5, metric="euclidean")
raw_knn.fit(X_train, y_train)

raw_query_prediction = raw_knn.predict(query)[0]
raw_distances, raw_positions = raw_knn.kneighbors(query, n_neighbors=5)
raw_neighbors = make_neighbor_table(X_train, y_train, raw_distances, raw_positions)

print("raw query prediction:", raw_query_prediction)
display(raw_neighbors)
display(raw_neighbors["neighbor_label"].value_counts().rename("votes"))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for label in CLASS_ORDER:
    mask = y_train == label
    ax.scatter(
        X_train.loc[mask, "practice_hours"],
        X_train.loc[mask, "assessment_score"],
        color=CLASS_COLORS[label],
        label=label,
        alpha=0.55,
    )
neighbor_rows = X_train.iloc[raw_positions[0]]
ax.scatter(
    neighbor_rows["practice_hours"],
    neighbor_rows["assessment_score"],
    s=190,
    facecolors="none",
    edgecolors="#202020",
    linewidths=1.6,
    label="5 raw-space neighbors",
)
ax.scatter(query["practice_hours"], query["assessment_score"], marker="*", s=260, color="#7570b3", label="query")
for _, row in raw_neighbors.iterrows():
    ax.annotate(row["case_id"], (row["practice_hours"], row["assessment_score"]), fontsize=8)
ax.set(title="Query and its raw-space neighborhood", xlabel="Practice hours", ylabel="Assessment score")
ax.legend(loc="best")
ax.grid(alpha=0.18)
plt.show()


### Observation checkpoint

Record the actual neighbor IDs, distance order and vote. If your prediction differed, identify the first geometric assumption that was wrong. Do not change the query after seeing the result.


## Prediction checkpoint 3 — vary k

Predict which `k` among 1, 3, 5, 9, 15 and 25 will:

- be most sensitive to one training point;
- produce the smoothest boundary;
- change the fixed query prediction;
- perform best on the unchanged test split.

**Your prediction:** _write here before running_.


In [ ]:
k_rows = []
for k in [1, 3, 5, 9, 15, 25]:
    model = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    model.fit(X_train, y_train)
    k_rows.append(
        {
            "k": k,
            "test_accuracy": accuracy_score(y_test, model.predict(X_test)),
            "query_prediction": model.predict(query)[0],
        }
    )
k_results = pd.DataFrame(k_rows)
display(k_results)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_results["k"], k_results["test_accuracy"], marker="o", color="#1b9e77")
for _, row in k_results.iterrows():
    ax.annotate(row["query_prediction"], (row["k"], row["test_accuracy"]), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
ax.set(xlabel="k", ylabel="Test accuracy", ylim=(0.4, 1.04), title="Same split, different neighborhood size")
ax.grid(alpha=0.22)
plt.show()


### Explain k, do not merely tune it

Small `k` makes a vote highly local and potentially unstable. Large `k` averages over more distant cases and can erase a useful local pattern. Record where the query prediction changes and explain it by inspecting which additional labels enter the vote.


## Prediction checkpoint 4 — distance metric

Euclidean distance draws circular neighborhoods in two dimensions; Manhattan distance draws diamond-shaped neighborhoods. For a boundary probe at `(2.65, 62.0)`, predict whether the same five neighbors and class survive when only the metric changes.

**Your prediction:** _write here before running_.


In [ ]:
metric_query = pd.DataFrame([[2.65, 62.0]], columns=informative_features)
metric_rows = []
metric_neighbors = {}
for metric in ["euclidean", "manhattan"]:
    model = KNeighborsClassifier(n_neighbors=5, metric=metric).fit(X_train, y_train)
    distances, positions = model.kneighbors(metric_query, n_neighbors=5)
    table = make_neighbor_table(X_train, y_train, distances, positions)
    metric_neighbors[metric] = set(table["case_id"])
    metric_rows.append(
        {
            "metric": metric,
            "test_accuracy": accuracy_score(y_test, model.predict(X_test)),
            "query_prediction": model.predict(metric_query)[0],
            "neighbor_ids": ", ".join(table["case_id"]),
        }
    )
metric_results = pd.DataFrame(metric_rows)
neighbor_overlap = len(metric_neighbors["euclidean"] & metric_neighbors["manhattan"])
display(metric_results)
print(f"Neighbor overlap: {neighbor_overlap}/5")


## Prediction checkpoint 5 — scale and inverse-scale

`assessment_score` spans more raw units than `practice_hours`. Predict:

1. which coordinate dominates raw Euclidean distances;
2. whether standardization changes the query's neighbors or class;
3. whether inverse-transforming the scaled query will recover exactly `(6.0, 67.0)`.

**Your prediction:** _write here before running_.


In [ ]:
scaled_knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5, metric="euclidean"),
)
scaled_knn.fit(X_train, y_train)

scaled_query_prediction = scaled_knn.predict(query)[0]
scaled_test_accuracy = accuracy_score(y_test, scaled_knn.predict(X_test))
scaler = scaled_knn.named_steps["standardscaler"]
scaled_classifier = scaled_knn.named_steps["kneighborsclassifier"]
query_scaled = scaler.transform(query)
scaled_distances, scaled_positions = scaled_classifier.kneighbors(query_scaled, n_neighbors=5)
scaled_neighbors = make_neighbor_table(X_train, y_train, scaled_distances, scaled_positions)

query_round_trip = pd.DataFrame(scaler.inverse_transform(query_scaled), columns=informative_features)
round_trip_max_error = float(np.abs(query_round_trip - query).to_numpy().max())

print("raw prediction:", raw_query_prediction, "scaled prediction:", scaled_query_prediction)
print("raw accuracy:", round(accuracy_score(y_test, raw_knn.predict(X_test)), 3))
print("scaled accuracy:", round(scaled_test_accuracy, 3))
print("fitted training means:", dict(zip(informative_features, scaler.mean_.round(3))))
print("fitted training scales:", dict(zip(informative_features, scaler.scale_.round(3))))
print("inverse-transform max error:", round_trip_max_error)
display(scaled_neighbors)
display(query_round_trip)


In [ ]:
X_train_scaled = scaler.transform(X_train)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for label in CLASS_ORDER:
    mask = (y_train == label).to_numpy()
    axes[0].scatter(X_train.loc[mask, "practice_hours"], X_train.loc[mask, "assessment_score"], color=CLASS_COLORS[label], label=label, alpha=0.65)
    axes[1].scatter(X_train_scaled[mask, 0], X_train_scaled[mask, 1], color=CLASS_COLORS[label], label=label, alpha=0.65)
axes[0].scatter(query.iloc[0, 0], query.iloc[0, 1], marker="*", s=220, color="#7570b3")
axes[1].scatter(query_scaled[0, 0], query_scaled[0, 1], marker="*", s=220, color="#7570b3")
axes[0].set(title="Raw units", xlabel="Practice hours", ylabel="Assessment score")
axes[1].set(title="Training-standardized units", xlabel="Practice hours (z)", ylabel="Assessment score (z)")
for ax in axes:
    ax.grid(alpha=0.18)
axes[0].legend()
plt.tight_layout()
plt.show()


### Scaling observation

Compare the raw and standardized neighbor IDs. Name the cases that entered or left and connect their labels to the changed vote. Standardization changes units; it does not prove that a feature is relevant.


## Controlled failure — a weak high-scale feature dominates

`interface_event_count` was generated independently of the target. Adding it should not define meaningful similarity, but its differences are measured in thousands.

Follow: **symptom → hypothesis → smallest test → distance decomposition → repair → verification**.


## Prediction checkpoint 6 — seed the failure

Before adding the weak feature, predict:

- its share of raw squared Euclidean distance;
- whether test accuracy will rise, hold or collapse;
- whether the query at `interface_event_count=9200` will keep its scaled informative-space class;
- what neighbor property will replace learning-behavior similarity.

**Your prediction:** _write here before running_.


In [ ]:
polluted_features = informative_features + [weak_feature]
X_polluted_train = data.loc[X_train.index, polluted_features]
X_polluted_test = data.loc[X_test.index, polluted_features]
query_polluted = pd.DataFrame([[6.0, 67.0, 9200]], columns=polluted_features)

polluted_raw_knn = KNeighborsClassifier(n_neighbors=5, metric="euclidean").fit(X_polluted_train, y_train)
polluted_raw_accuracy = accuracy_score(y_test, polluted_raw_knn.predict(X_polluted_test))
polluted_raw_prediction = polluted_raw_knn.predict(query_polluted)[0]
polluted_distances, polluted_positions = polluted_raw_knn.kneighbors(query_polluted, n_neighbors=5)
polluted_neighbors = make_neighbor_table(X_polluted_train, y_train, polluted_distances, polluted_positions)

feature_ranges = data[polluted_features].max() - data[polluted_features].min()
print("raw feature ranges:")
display(feature_ranges.rename("range"))
print("polluted raw test accuracy:", round(polluted_raw_accuracy, 3))
print("polluted raw query prediction:", polluted_raw_prediction)
display(polluted_neighbors)


In [ ]:
closest_position = polluted_positions[0, 0]
closest_raw = X_polluted_train.iloc[closest_position]
squared_contributions = (closest_raw - query_polluted.iloc[0]) ** 2
contribution_share = squared_contributions / squared_contributions.sum()
distance_diagnosis = pd.DataFrame(
    {
        "query_value": query_polluted.iloc[0],
        "neighbor_value": closest_raw,
        "squared_contribution": squared_contributions,
        "share_of_squared_distance": contribution_share,
    }
)
weak_distance_share = float(contribution_share[weak_feature])
display(distance_diagnosis)
print(f"Weak feature share of squared distance: {weak_distance_share:.6%}")


## Prediction checkpoint 7 — repair and verify

Predict the outcome of two controlled repairs:

1. standardize all three features using training-only statistics;
2. standardize only the two informative features and exclude the weak feature.

Which should recover the score most? Which design is easier to justify? Fix your answer before running.

**Your prediction:** _write here before running_.


In [ ]:
polluted_scaled_knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5, metric="euclidean"),
).fit(X_polluted_train, y_train)

polluted_scaled_accuracy = accuracy_score(y_test, polluted_scaled_knn.predict(X_polluted_test))
polluted_scaled_prediction = polluted_scaled_knn.predict(query_polluted)[0]

polluted_scaler = polluted_scaled_knn.named_steps["standardscaler"]
polluted_classifier = polluted_scaled_knn.named_steps["kneighborsclassifier"]
polluted_scaled_query = polluted_scaler.transform(query_polluted)
repair_distances, repair_positions = polluted_classifier.kneighbors(polluted_scaled_query, n_neighbors=5)
repair_neighbors = make_neighbor_table(X_polluted_train, y_train, repair_distances, repair_positions)

repair_comparison = pd.DataFrame(
    [
        {"design": "raw informative", "test_accuracy": accuracy_score(y_test, raw_knn.predict(X_test)), "query_prediction": raw_query_prediction},
        {"design": "scaled informative", "test_accuracy": scaled_test_accuracy, "query_prediction": scaled_query_prediction},
        {"design": "raw + weak high-scale", "test_accuracy": polluted_raw_accuracy, "query_prediction": polluted_raw_prediction},
        {"design": "scaled + weak feature", "test_accuracy": polluted_scaled_accuracy, "query_prediction": polluted_scaled_prediction},
    ]
)
display(repair_comparison)
display(repair_neighbors)


### Repair explanation

Scaling prevents numeric units from automatically dominating, but it cannot make an irrelevant feature useful. Compare the scaled three-feature model with the scaled informative-only model before deciding whether to keep the instrumentation feature.


## Prediction checkpoint 8 — boundary and prediction changes

Sketch the raw and standardized `k=5` boundaries. Circle where you expect predictions to differ and explain which feature's unit choice causes the change.

**Your prediction:** _write here before running_.


In [ ]:
def plot_boundary(ax, model, features, labels, title):
    x_min, x_max = features.iloc[:, 0].min() - 0.5, features.iloc[:, 0].max() + 0.5
    y_min, y_max = features.iloc[:, 1].min() - 2.0, features.iloc[:, 1].max() + 2.0
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 180), np.linspace(y_min, y_max, 180))
    grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=informative_features)
    encoded = pd.Series(model.predict(grid)).map(CLASS_TO_INT).to_numpy().reshape(xx.shape)
    ax.contourf(xx, yy, encoded, alpha=0.22, cmap=ListedColormap([CLASS_COLORS[label] for label in CLASS_ORDER]))
    for label in CLASS_ORDER:
        mask = labels == label
        ax.scatter(features.loc[mask, informative_features[0]], features.loc[mask, informative_features[1]], color=CLASS_COLORS[label], s=28, alpha=0.72)
    ax.scatter(query.iloc[0, 0], query.iloc[0, 1], marker="*", s=230, color="#7570b3", edgecolor="white")
    ax.set(title=title, xlabel="Practice hours", ylabel="Assessment score")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_boundary(axes[0], raw_knn, X_train, y_train, f"Raw units — query: {raw_query_prediction}")
plot_boundary(axes[1], scaled_knn, X_train, y_train, f"Standardized — query: {scaled_query_prediction}")
plt.tight_layout()
plt.show()


### Boundary observation

Record one region that changed class. Pick a point in that region, retrieve both neighbor sets, and explain the change as membership plus votes—not as “the plot moved.”


## Code reading trace

Before running the next cell, trace:

`raw query → fitted training scaler → standardized query → stored standardized cases → neighbor indices → original row labels → vote`

Identify which pipeline step learns `mean_` and `scale_`, which stores `_fit_X` and `_y`, and why passing an unscaled query directly to the inner classifier would be a contract violation.


In [ ]:
pipeline_trace = pd.DataFrame(
    [
        {
            "step": name,
            "type": type(step).__name__,
            "learned_state_examples": "mean_, scale_" if name == "standardscaler" else "_fit_X, _y",
        }
        for name, step in scaled_knn.named_steps.items()
    ]
)
display(pipeline_trace)
print("raw query shape:", query.shape)
print("transformed query shape:", query_scaled.shape)
print("stored training matrix shape:", scaled_classifier._fit_X.shape)


## ADR — define similarity for V03

Use `missions/M13/adr_prompt.md` and `templates/ADR.md` to decide among raw Euclidean, standardized Euclidean, standardized Manhattan, removal of the weak feature, or a non-KNN alternative.

Include neighbor-level evidence, trade-offs, safeguards and revisit conditions. A leaderboard score is not a sufficient decision record.


## No-AI Gate

Complete `missions/M13/no_ai_gate.md` without AI-generated code. Hand-calculate distances, implement neighbor ordering and voting, seed a scale mismatch, repair it with training-only statistics and explain every neighborhood change.


## Learner evidence — intentionally blank

Fill after completing the work; do not replace observations with claims.

- Prediction log:
- Query neighbor trace:
- k explanation:
- Metric explanation:
- Scaling round trip:
- Controlled-failure hypothesis:
- Per-feature distance evidence:
- Repair verification:
- Boundary explanation:
- Code-reading trace:
- No-AI transfer evidence:
- ADR path:


In [ ]:
assert len(data) == 96
assert data[target].value_counts().to_dict() == {"independent": 48, "guided": 48}
assert round_trip_max_error < 1e-10
assert raw_query_prediction != scaled_query_prediction
assert weak_distance_share > 0.99
assert polluted_raw_accuracy <= 0.65
assert polluted_scaled_accuracy >= 0.85
assert polluted_scaled_accuracy - polluted_raw_accuracy >= 0.25
print("M13 executable contract checks passed.")


## Completion reflection

Explain in plain language:

1. what KNN stores during fit;
2. how `k` changes local sensitivity;
3. how metric and units define near;
4. why training-only scaling matters;
5. why scaling does not establish relevance;
6. how a changed feature space changes neighbors, votes, boundaries and predictions;
7. when V03 should prefer another model.
